# P01 — Construcción y evaluación de un modelo de riesgo

**Caso: Velora Finance**

El código ya está preparado. El alumno solo tiene que ejecutar las celdas, observar los resultados y discutir su significado. La única modificación propuesta es cambiar un valor de `THRESHOLD_ELEGIDO`.

`default = 0`: no hubo impago.  
`default = 1`: hubo impago.


## 1. Preparación

Esta celda carga las librerías utilizadas para manipular datos, entrenar el modelo, calcular métricas y representar resultados. No es necesario memorizar los imports.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import urllib.request

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    confusion_matrix, ConfusionMatrixDisplay, accuracy_score,
    precision_score, recall_score, f1_score, roc_auc_score,
    average_precision_score, roc_curve, precision_recall_curve, auc
)

pd.set_option("display.max_columns", None)
print("Entorno preparado.")


**Resultado esperado:** `Entorno preparado.` significa que podemos continuar.


## 2. Cargar el dataset

El dataset se genera de forma reproducible a partir del script versionado en GitHub, de modo que todos trabajan con los mismos datos.


In [ ]:
GENERATOR_URL = "https://raw.githubusercontent.com/pacosepulveda/ia-confiable-explicable-labs/main/resources/generate_velora_credit.py"
namespace = {"__name__": "velora_generator"}
with urllib.request.urlopen(GENERATOR_URL) as response:
    exec(response.read().decode("utf-8"), namespace)
df = namespace["generate_dataset"]()
print("Dataset cargado.")
display(df.head())


**Cómo interpretar el resultado:** cada fila es una solicitud de crédito. `application_id` es un identificador. `customer_group` se conserva para prácticas posteriores, pero no se utiliza como predictor en P01. `default` contiene el resultado real.


## 3. Revisar los datos

Antes de entrenar comprobamos tamaño, tipos, valores ausentes y distribución de la variable objetivo.


In [ ]:
print("Dimensiones:", df.shape)
print()
df.info()


**Resultado esperado:** 3.000 solicitudes, 12 columnas y sin valores nulos. Así podemos centrarnos en la evaluación del modelo y no en limpieza de datos.


In [ ]:
df.describe()


**Cómo interpretarlo:** las variables están en escalas muy diferentes: ingresos e importes monetarios, ratios, recuentos y meses. Por eso utilizaremos `StandardScaler` antes de la regresión logística.


## 4. ¿Está equilibrada la variable objetivo?


In [ ]:
counts = df["default"].value_counts().sort_index()
percentages = df["default"].value_counts(normalize=True).sort_index() * 100
display(pd.DataFrame({"Número de casos": counts, "Porcentaje": percentages}))
print(f"Tasa de impago: {df['default'].mean():.2%}")


**Cómo interpretar el resultado:** la tasa de impago es aproximadamente del **11,6 %**. La clase `default=1` es minoritaria. Esto importa porque una accuracy alta puede ocultar un mal comportamiento sobre la clase que más nos preocupa.


## 5. Preparar entrenamiento y test

Usamos el 80 % para entrenar y el 20 % para evaluar. `stratify=y` mantiene aproximadamente la misma proporción de impagos en ambos conjuntos.


In [ ]:
FEATURES = ["age", "annual_income", "debt_ratio", "employment_months",
            "late_payments_12m", "credit_history_months", "existing_debt",
            "requested_amount", "savings_balance"]
X = df[FEATURES]
y = df["default"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print("Train:", len(X_train), "solicitudes")
print("Test :", len(X_test), "solicitudes")
print(f"Impago global: {y.mean():.2%}")
print(f"Impago train : {y_train.mean():.2%}")
print(f"Impago test  : {y_test.mean():.2%}")


**Resultado esperado:** 2.400 casos de entrenamiento y 600 de test, con tasas de impago prácticamente iguales.


## 6. Entrenar el modelo base

Usamos regresión logística porque produce probabilidades, es rápida y sirve como línea base clara para analizar el comportamiento del sistema.


In [ ]:
model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000, random_state=42))
])
model.fit(X_train, y_train)
print("Modelo entrenado correctamente.")


## 7. Probabilidad frente a decisión

El modelo produce una probabilidad estimada de impago. La clasificación `0/1` aparece después al aplicar un **threshold**. Empezamos con `0.5`.


In [ ]:
proba = model.predict_proba(X_test)[:, 1]
threshold = 0.5
y_pred = (proba >= threshold).astype(int)
display(pd.DataFrame({
    "probabilidad_impago": proba[:10],
    "prediccion": y_pred[:10],
    "resultado_real": y_test.iloc[:10].values
}))


**Idea clave:** el modelo no decide directamente aprobar o rechazar. Genera probabilidades; la organización define cómo convertirlas en decisiones.


## 8. Primera impresión: Accuracy


In [ ]:
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.2%}")


**Cómo interpretar el resultado:** la accuracy ronda el 90 %, lo que parece muy bueno. Pero todavía no sabemos qué tipos de errores comete el modelo. Con una clase minoritaria, la accuracy por sí sola puede resultar engañosa.


## 9. Matriz de confusión


In [ ]:
cm = confusion_matrix(y_test, y_pred)
ConfusionMatrixDisplay(cm, display_labels=["No impago", "Impago"]).plot()
plt.title("Matriz de confusión — threshold = 0.5")
plt.show()
tn, fp, fn, tp = cm.ravel()
print("TN:", tn, " FP:", fp, " FN:", fn, " TP:", tp)


**Cómo interpretar el resultado:** un **falso negativo** es una operación clasificada como de bajo riesgo que termina en impago. Un **falso positivo** es una operación clasificada como riesgo que realmente no termina en impago. Sus consecuencias son diferentes, por lo que no basta con contar aciertos totales.


## 10. Precision, Recall y F1


In [ ]:
precision = precision_score(y_test, y_pred, zero_division=0)
recall = recall_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)
display(pd.DataFrame({
    "Métrica": ["Accuracy", "Precision", "Recall", "F1"],
    "Valor": [accuracy, precision, recall, f1]
}))


**Interpretación:** `Precision` indica cuántos de los casos marcados como impago realmente incumplen. `Recall` indica cuántos de todos los impagos reales conseguimos detectar. Con `threshold=0.5`, el Recall queda alrededor del **20–25 %**, aunque la accuracy esté cerca del 90 %. Esta es una de las conclusiones principales de P01.


## 11. Comparar thresholds

El código calcula automáticamente el efecto de varios umbrales.


In [ ]:
def evaluate_threshold(y_true, probabilities, threshold):
    pred = (probabilities >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, pred).ravel()
    return {
        "threshold": threshold,
        "accuracy": accuracy_score(y_true, pred),
        "precision": precision_score(y_true, pred, zero_division=0),
        "recall": recall_score(y_true, pred, zero_division=0),
        "f1": f1_score(y_true, pred, zero_division=0),
        "fp": fp, "fn": fn, "tp": tp, "tn": tn
    }
thresholds = [0.20, 0.30, 0.40, 0.50, 0.60, 0.70]
threshold_results = pd.DataFrame([evaluate_threshold(y_test, proba, t) for t in thresholds])
display(threshold_results.round(3))


**Cómo interpretar la tabla:** al bajar el threshold aumenta el Recall y disminuyen los falsos negativos, pero aumentan los falsos positivos. Al subirlo ocurre lo contrario. No existe un threshold universalmente correcto: depende del coste y del riesgo asociado a cada tipo de error.


## 12. Única modificación propuesta al alumno

Cambia solamente `THRESHOLD_ELEGIDO` y vuelve a ejecutar la celda. Prueba, por ejemplo, `0.20`, `0.30`, `0.50` o `0.60`.


In [ ]:
THRESHOLD_ELEGIDO = 0.30
resultado = evaluate_threshold(y_test, proba, THRESHOLD_ELEGIDO)
for clave in ["accuracy", "precision", "recall", "f1", "fp", "fn"]:
    print(f"{clave:9}: {resultado[clave]:.3f}" if isinstance(resultado[clave], float) else f"{clave:9}: {resultado[clave]}")


**Discusión:** si conceder crédito a un futuro impago fuese muy costoso, ¿usarías el mismo threshold que si el principal coste fuese rechazar clientes solventes? No necesariamente. El modelo y sus probabilidades no cambian; cambia la política que transforma esas probabilidades en decisiones.


## 13. ROC-AUC y Precision-Recall — extensión

Esta parte puede tratarse como demostración del instructor. Evalúa el modelo considerando muchos thresholds posibles.


In [ ]:
roc_auc = roc_auc_score(y_test, proba)
avg_precision = average_precision_score(y_test, proba)
pr_precision, pr_recall, _ = precision_recall_curve(y_test, proba)
pr_auc = auc(pr_recall, pr_precision)
print(f"ROC-AUC: {roc_auc:.3f}")
print(f"Average Precision: {avg_precision:.3f}")
print(f"PR-AUC: {pr_auc:.3f}")

fpr, tpr, _ = roc_curve(y_test, proba)
plt.figure(figsize=(7,5))
plt.plot(fpr, tpr, label=f"ROC-AUC = {roc_auc:.3f}")
plt.plot([0,1],[0,1],"--", label="Clasificador aleatorio")
plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate"); plt.legend(); plt.show()

precision_curve, recall_curve, _ = precision_recall_curve(y_test, proba)
plt.figure(figsize=(7,5))
plt.plot(recall_curve, precision_curve, label=f"Average Precision = {avg_precision:.3f}")
plt.axhline(y_test.mean(), linestyle="--", label=f"Prevalencia = {y_test.mean():.3f}")
plt.xlabel("Recall"); plt.ylabel("Precision"); plt.legend(); plt.show()


**Interpretación:** ROC-AUC resume la capacidad de ordenar positivos por encima de negativos a través de distintos thresholds. Precision-Recall resulta especialmente útil con clases desequilibradas. Aun así, una buena métrica global no determina qué threshold debe utilizar una organización.


## 14. Revisar errores concretos


In [ ]:
results = df.loc[X_test.index].copy()
results["prob_default"] = proba
results["prediction"] = y_pred
fp_cases = results[(results["default"] == 0) & (results["prediction"] == 1)]
fn_cases = results[(results["default"] == 1) & (results["prediction"] == 0)]
cols = ["application_id", "annual_income", "debt_ratio", "employment_months",
        "late_payments_12m", "requested_amount", "prob_default", "prediction", "default"]
print("Ejemplo de falso positivo:")
display(fp_cases[cols].head(1))
print("Ejemplo de falso negativo:")
display(fn_cases[cols].head(1))


**Cómo interpretar estos casos:** ahora vemos errores concretos, pero todavía no sabemos explicar por qué el modelo asignó exactamente esas probabilidades. Esa pregunta dará paso a la práctica de explicabilidad.

# Conclusiones de P01

1. Una accuracy alta no garantiza un comportamiento adecuado.
2. Falsos positivos y falsos negativos tienen consecuencias diferentes.
3. El modelo devuelve probabilidades; la decisión binaria aparece al aplicar un threshold.
4. Cambiar el threshold modifica el equilibrio entre tipos de error.
5. La evaluación de un sistema debe relacionarse con el contexto y el riesgo de la decisión.

La siguiente pregunta será: **¿funciona igual para todos los grupos?**
